# Keeping English intact while adapting to Vietnamese
LoRA fine-tuning of `meta-llama/Llama-3.2-1B-Instruct` on OPUS-100 en-vi, comparing plain fine-tuning, plain alignment, JEPA (live / anchored) and 10% replay. Evaluated on FLORES en↔vi (BLEU, chrF++), MLQA-vi (EM, F1) and IFEval (English instruction following).

**How to use:** run the 4 setup cells once per session, then run any arm's two cells (**cell 1 = train, cell 2 = eval**). Arms are independent, so run one or several per session. Finished arms are skipped when you re-run them.

**Fixed for every arm:** seed 42, 50,000 OPUS-100 en-vi pairs (HF Hub, shuffled with seed 42), LoRA r=16 / alpha=32 on all 7 projections, effective batch 16 (4 x 4), lr 2e-4, fp16, MAX_LEN 256 (pairs over the limit are dropped, never truncated), λ=0.5 for every JEPA/alignment arm.


| Arm | `train.py` arguments that change the method |
|---|---|
| Base model | none (eval only) |
| Plain fine-tuning | `--lbd 0` |
| Plain alignment | `--lbd 0.5 --k 0 --target live` |
| JEPA live | `--lbd 0.5 --k 1 --target live` |
| JEPA anchored, k=0 | `--lbd 0.5 --k 0 --target base` |
| JEPA anchored, k=1 | `--lbd 0.5 --k 1 --target base` |
| JEPA anchored, k=1, fixed [PRED] | `--lbd 0.5 --k 1 --target base --pred_fix` |
| Replay 10% | `--lbd 0 --replay_frac 0.10` (45k OPUS + 5k Alpaca) |
| Replay 10% + JEPA anchored, k=1 | `--lbd 0.5 --k 1 --target base --replay_frac 0.10`  |

## Setup 1/4: packages

In [ ]:
!pip -q install transformers==5.20.2 peft==0.21.2 sacrebleu==2.6.0 datasets accelerate huggingface_hub
!pip -q install -U "torchao>=0.16.0"
!pip -q install langdetect immutabledict "nltk>=3.9.1"

import torch, transformers, peft, datasets, huggingface_hub
print("PyTorch:", torch.__version__)
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (Runtime > Change runtime type > T4 GPU)")
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Datasets:", datasets.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)

## Setup 2/4: Google Drive, Hugging Face login, GitHub repo (eval data)

In [ ]:
import os
import subprocess
from google.colab import drive, userdata
from huggingface_hub import login

# Google Drive: adapters, telemetry and eval results are saved here
drive.mount("/content/drive")
ADAPTER_ROOT = "/content/drive/MyDrive/adapters"
os.makedirs(ADAPTER_ROOT, exist_ok=True)

# Hugging Face login
try:
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    print("No HF_TOKEN secret found -> interactive login")
    login()

# GitHub repo-holds the eval sets in data

REPO_URL = "https://github.com/Tanay-s5/sift.git"
REPO_DIR = "/content/repo"

if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
DATA_DIR = f"{REPO_DIR}/data"
for f in ("flores.json", "mlqa_vi.jsonl", "ifeval.jsonl"):
    assert os.path.exists(f"{DATA_DIR}/{f}"), f"{DATA_DIR}/{f} not found: put the eval files in the repo's data/ folder"
print("Drive adapters folder:", ADAPTER_ROOT)
print("Eval data folder     :", DATA_DIR)

## Setup 3/4: training script (`train.py`, one script for every arm)

In [ ]:
%%writefile train.py
# ======================================================================================
# train.py : one training script for every arm. Only the ARGUMENTS change.
#
#   --lbd 0                         plain LM fine-tuning
#   --lbd 0.5 --k 0 --target live   plain alignment (VI last-token vector pulled to the EN one, no [PRED] token)
#   --lbd 0.5 --k 1 --target live   JEPA live   (target comes from the model being trained)
#   --lbd 0.5 --k 0/1 --target base JEPA anchored (target comes from the frozen base model, adapter off)
#   --pred_fix                      k=1 only: mean-init the [PRED] embedding row and train just that row
#   --replay_frac 0.10 --lbd 0      plain LM with 10% Alpaca replay (45k OPUS + 5k Alpaca)
#   --replay_frac 0.10 --lbd 0.5 --k 1 --target base   anchored JEPA + replay
#
# Data: OPUS-100 en-vi train split from the Hugging Face Hub,
# shuffled with seed 42, first 50,000 pairs. pair is dropped if over token length.
# ======================================================================================
import os
import sys
import json
import time
import csv
import hashlib
import argparse
import random
import contextlib
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    get_cosine_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, TaskType

args = None
ARM_NAME = OUT = MODEL_DIR = None
MAX_LEN = BS = ACCUM = LR = None
USE_FP16 = AMP_DTYPE = None
DATA_INFO = {}


# CLI ARGUMENTS

def get_args(argv=None):
    parser = argparse.ArgumentParser(description="One training script for all arms (plain LM / alignment / JEPA live / JEPA anchored / replay)")
    parser.add_argument("--k", type=int, default=1, choices=[0, 1], help="0 = no [PRED] token, 1 = append the [PRED] token to the Vietnamese view")
    parser.add_argument("--lbd", type=float, default=0.5, help="JEPA/alignment loss weight lambda. 0.0 = plain LM fine-tuning (JEPA passes are skipped)")
    parser.add_argument("--gamma", type=float, default=1.0, help="LM loss weight gamma")
    parser.add_argument("--layer", type=int, default=-1, help="Layer index for the representation (-1 = last)")
    parser.add_argument("--target", type=str, default="base", choices=["live", "base"],
                        help="'base' = ANCHORED: English target from the frozen base model (adapter off, no gradient). "
                             "'live' = target from the model being trained.")
    parser.add_argument("--pred_fix", action="store_true",
                        help="k=1 only: initialise the [PRED] embedding row to the mean embedding and make only that row trainable")
    parser.add_argument("--pred_token", type=str, default="<|reserved_special_token_0|>", help="Token used as [PRED]")
    parser.add_argument("--replay_frac", type=float, default=0.0,
                        help="Fraction of n_total that is Alpaca replay (0.10 -> 45k OPUS + 5k Alpaca). With --lbd > 0 the JEPA loss is computed on the OPUS rows only")
    parser.add_argument("--replay_dataset", type=str, default="yahma/alpaca-cleaned", help="HF dataset used for replay")
    parser.add_argument("--seed", type=int, default=42, help="Random seed (data subset, shuffling, init)")
    parser.add_argument("--arm_name", type=str, default=None, help="Output folder name inside --out_root")
    parser.add_argument("--bs", type=int, default=4, help="Micro-batch size through the GPU")
    parser.add_argument("--accum", type=int, default=4, help="Gradient accumulation steps; effective batch = bs * accum")
    parser.add_argument("--max_len", type=int, default=256, help="Token limit for the LM sequence and both views; a pair is dropped if any exceeds it")
    parser.add_argument("--n_total", type=int, default=50_000, help="Size of the OPUS-100 subset (replay arms: OPUS + replay = n_total)")
    parser.add_argument("--lr", type=float, default=2e-4, help="Learning rate")
    parser.add_argument("--precision", type=str, default="fp16", choices=["fp16", "bf16"], help="Mixed precision. fp16 on a T4")
    parser.add_argument("--model", type=str, default="meta-llama/Llama-3.2-1B-Instruct", help="HF model id or local path")
    parser.add_argument("--opus_dataset", type=str, default="Helsinki-NLP/opus-100", help="HF dataset id for the training pairs")
    parser.add_argument("--opus_config", type=str, default="en-vi", help="HF dataset config")
    parser.add_argument("--opus_json", type=str, default=None,
                        help="Optional: read the first n_total pairs from a local json (list of {en, vi}) instead of the HF Hub")
    parser.add_argument("--out_root", type=str, default="/content/drive/MyDrive/adapters", help="Each arm is saved to <out_root>/<arm_name>")
    parser.add_argument("--overwrite", action="store_true", help="Retrain even if <arm>/ckpt_100pct already exists")
    return parser.parse_args(argv)


def default_arm_name(a):
    if a.lbd == 0:
        return f"replay_{round(a.replay_frac * 100)}pct" if a.replay_frac > 0 else "plain_fine_tuning"
    name = "jepa_anchored" if a.target == "base" else "jepa_live"
    if a.k == 0:
        name += "_k0"
    if a.pred_fix:
        name += "_new_pred"
    if a.replay_frac > 0:
        name = f"replay_{round(a.replay_frac * 100)}pct_{name}"
    return name


def configure(argv=None):
    global args, ARM_NAME, OUT, MODEL_DIR, MAX_LEN, BS, ACCUM, LR, USE_FP16, AMP_DTYPE
    args = get_args(argv)
    if args.bs < 1 or args.accum < 1:
        raise ValueError("--bs and --accum must be positive.")
    if not 0.0 <= args.replay_frac < 1.0:
        raise ValueError("--replay_frac must be in [0, 1).")
    if args.pred_fix and not (args.k == 1 and args.lbd > 0):
        raise SystemExit("--pred_fix only makes sense with --k 1 and --lbd > 0.")
    ARM_NAME = args.arm_name or default_arm_name(args)
    OUT = os.path.join(args.out_root, ARM_NAME)
    MODEL_DIR = args.model
    MAX_LEN, BS, ACCUM, LR = args.max_len, args.bs, args.accum, args.lr
    USE_FP16 = args.precision == "fp16"
    AMP_DTYPE = torch.float16 if USE_FP16 else torch.bfloat16
    random.seed(args.seed)
    torch.manual_seed(args.seed)



#  TOKENIZATION

def _hf_load(name, config=None):
    try:
        return load_dataset(name, config, split="train") if config else load_dataset(name, split="train")
    except Exception as e:
        print(f"load_dataset({name}) failed ({type(e).__name__}: {e}) -> retrying with revision='refs/convert/parquet'")
        if config:
            return load_dataset(name, config, split="train", revision="refs/convert/parquet")
        return load_dataset(name, split="train", revision="refs/convert/parquet")


def load_opus_pairs(n_samples):
    pairs = []
    if args.opus_json:
        print(f"Loading OPUS-100 pairs from local file: {args.opus_json}")
        with open(args.opus_json, "r", encoding="utf-8") as f:
            data = json.load(f)
        data = data if isinstance(data, list) else data.get("train", [])
        for item in data[:n_samples]:
            en = item.get("en") or item.get("english") or item.get("translation", {}).get("en")
            vi = item.get("vi") or item.get("vietnamese") or item.get("translation", {}).get("vi")
            if en and vi:
                pairs.append((str(en), str(vi)))
        source = f"local:{args.opus_json}"
    else:
        print(f"Loading {args.opus_dataset} [{args.opus_config}] from the Hugging Face Hub ...")
        ds = _hf_load(args.opus_dataset, args.opus_config)
        ds = ds.shuffle(seed=args.seed).select(range(min(n_samples, len(ds))))
        for ex in ds:
            t = ex["translation"]
            if t.get("en") and t.get("vi"):
                pairs.append((str(t["en"]), str(t["vi"])))
        source = f"hf:{args.opus_dataset}/{args.opus_config} train, shuffle(seed={args.seed}), first {n_samples}"
    md5 = hashlib.md5("\n".join(f"{en}\t{vi}" for en, vi in pairs).encode("utf-8")).hexdigest()
    DATA_INFO.update({"opus_source": source, "opus_subset_pairs": len(pairs), "opus_subset_md5": md5})
    print(f"OPUS subset: {len(pairs)} pairs | md5 {md5}\n  first pair: {pairs[0]}")
    return pairs


def chat(tok, text):
    return tok.apply_chat_template([{"role": "user", "content": text}], tokenize=False, add_generation_prompt=True)


def encode_row(tok, user_lm, ans_lm, pred_id=None, k=1, need_jepa=False, en=None, vi=None):
    p_ids = tok(chat(tok, user_lm), add_special_tokens=False).input_ids
    full_ids = tok(
        tok.apply_chat_template(
            [{"role": "user", "content": user_lm}, {"role": "assistant", "content": ans_lm}],
            tokenize=False
        ),
        add_special_tokens=False
    ).input_ids
    full_labels = [-100] * len(p_ids) + full_ids[len(p_ids):]

    too_long = len(full_ids) > MAX_LEN
    ctx_ids, tgt_ids = [], []
    if en is not None:
        # JEPA views: VI (context and [PRED]) and EN (target). VI is context, EN is target.
        ctx = tok(chat(tok, vi), add_special_tokens=False).input_ids
        tgt = tok(chat(tok, en), add_special_tokens=False).input_ids
        # The same filter for every arm
        too_long = too_long or (len(ctx) + 1 > MAX_LEN) or (len(tgt) > MAX_LEN)
        if need_jepa:
            ctx_ids = ctx + ([pred_id] if k == 1 else [])
            tgt_ids = tgt
    if too_long:
        return None
    return {"full_ids": full_ids, "full_labels": full_labels, "ctx_ids": ctx_ids, "tgt_ids": tgt_ids}


def load_replay_rows(tok, n_replay):
    print(f"Loading replay data {args.replay_dataset} from the Hugging Face Hub ...")
    ds = _hf_load(args.replay_dataset).shuffle(seed=args.seed)
    rows = []
    for ex in ds:
        instr = (ex.get("instruction") or "").strip()
        inp = (ex.get("input") or "").strip()
        out = (ex.get("output") or "").strip()
        if not instr or not out:
            continue
        enc = encode_row(tok, instr + (f"\n\n{inp}" if inp else ""), out)
        if enc is not None:
            rows.append(enc)
        if len(rows) == n_replay:
            break
    if len(rows) < n_replay:
        raise RuntimeError(f"Only {len(rows)} replay rows passed the length filter, needed {n_replay}.")
    return rows


def build_data(tok, pred_id, k, need_jepa):
    n_replay = round(args.replay_frac * args.n_total)
    n_trans = args.n_total - n_replay
    pairs = load_opus_pairs(args.n_total)[:n_trans]

    items = []
    for i, (en, vi) in enumerate(pairs):
        if i % 2 == 0:
            u_msg = f"Translate the following text to Vietnamese. Output only the translation.\n\n{en}"
            a_msg = vi
        else:
            u_msg = f"Translate the following text to English. Output only the translation.\n\n{vi}"
            a_msg = en
        items.append((u_msg, a_msg, en, vi))

    random.shuffle(items)

    processed = []
    for u, a, en, vi in items:
        enc = encode_row(tok, u, a, pred_id, k, need_jepa, en=en, vi=vi)
        if enc:
            processed.append(enc)
    n_translation = len(processed)

    n_replay_used = 0
    if n_replay > 0:
        replay_rows = load_replay_rows(tok, n_replay)
        n_replay_used = len(replay_rows)
        processed += replay_rows
        random.shuffle(processed)

    DATA_INFO.update({
        "replay_frac": args.replay_frac,
        "replay_data": args.replay_dataset if n_replay > 0 else None,
        "n_translation_samples": n_translation,
        "n_replay_samples": n_replay_used,
    })
    print(f"Total processed samples (k={k}, JEPA active={need_jepa}): {len(processed)} "
          f"({n_translation} translation + {n_replay_used} replay)")
    return processed

def collate_jepa(rows, pad_token_id, need_jepa):
    # Collate Full LM
    max_full = max(len(r["full_ids"]) for r in rows)
    full_ids = torch.full((len(rows), max_full), pad_token_id)
    full_lab = torch.full((len(rows), max_full), -100)
    full_mask = torch.zeros((len(rows), max_full), dtype=torch.long)

    for i, r in enumerate(rows):
        full_ids[i, :len(r["full_ids"])] = torch.tensor(r["full_ids"])
        full_lab[i, :len(r["full_labels"])] = torch.tensor(r["full_labels"])
        full_mask[i, :len(r["full_ids"])] = 1

    batch = {
        "full_ids": full_ids.cuda(),
        "full_mask": full_mask.cuda(),
        "full_lab": full_lab.cuda()
    }

    jrows = [r for r in rows if r["ctx_ids"]] if need_jepa else []  # replay rows have no JEPA views
    batch["has_jepa"] = len(jrows) > 0
    if jrows:

        max_ctx = max(len(r["ctx_ids"]) for r in jrows)
        ctx_ids = torch.full((len(jrows), max_ctx), pad_token_id)
        ctx_mask = torch.zeros((len(jrows), max_ctx), dtype=torch.long)
        ctx_last_idx = []


        max_tgt = max(len(r["tgt_ids"]) for r in jrows)
        tgt_ids = torch.full((len(jrows), max_tgt), pad_token_id)
        tgt_mask = torch.zeros((len(jrows), max_tgt), dtype=torch.long)
        tgt_last_idx = []

        for i, r in enumerate(jrows):
            ctx_ids[i, :len(r["ctx_ids"])] = torch.tensor(r["ctx_ids"])
            ctx_mask[i, :len(r["ctx_ids"])] = 1
            ctx_last_idx.append(len(r["ctx_ids"]) - 1)

            tgt_ids[i, :len(r["tgt_ids"])] = torch.tensor(r["tgt_ids"])
            tgt_mask[i, :len(r["tgt_ids"])] = 1
            tgt_last_idx.append(len(r["tgt_ids"]) - 1)

        batch.update({
            "ctx_ids": ctx_ids.cuda(),
            "ctx_mask": ctx_mask.cuda(),
            "ctx_last_idx": torch.tensor(ctx_last_idx).cuda(),
            "tgt_ids": tgt_ids.cuda(),
            "tgt_mask": tgt_mask.cuda(),
            "tgt_last_idx": torch.tensor(tgt_last_idx).cuda()
        })


    return batch


# TRAINING

def compute_anisotropy_stats(z):
    normed = F.normalize(z.float(), dim=-1)
    std = normed.std(dim=0).mean().item()
    if len(normed) > 1:
        pair_cos = (normed @ normed.T).fill_diagonal_(0).sum().item() / (len(normed) * (len(normed) - 1))
    else:
        pair_cos = 0.0
    return std, pair_cos

def train(model, tok, pred_id):
    need_jepa = args.lbd > 0.0
    data = build_data(tok, pred_id, args.k, need_jepa)
    total_steps = len(data) // (BS * ACCUM)

    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
    sched = get_cosine_schedule_with_warmup(opt, int(0.03 * total_steps), total_steps)

    scaler = torch.amp.GradScaler("cuda", enabled=USE_FP16)  # no-op when bf16
    ck = {max(1, int(total_steps * f)): int(f * 100) for f in (0.25, 0.50, 0.75, 1.0)}

    f_log = open(f"{OUT}/train_log.csv", "w", newline="")
    w_log = csv.writer(f_log)
    w_log.writerow(["step", "loss", "lm_loss", "jepa_loss", "ctx_std", "ctx_pair", "tgt_std", "tgt_pair", "lr", "grad_norm"])

    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    idx = 0
    model.train()

    for step in range(1, total_steps + 1):
        tot_loss_accum = 0.0
        tot_lm_accum = 0.0
        tot_jepa_accum = 0.0
        ctx_std_accum, ctx_pair_accum = 0.0, 0.0
        tgt_std_accum, tgt_pair_accum = 0.0, 0.0
        lr = opt.param_groups[0]["lr"]

        for _ in range(ACCUM):
            batch = collate_jepa(data[idx:idx + BS], tok.pad_token_id, need_jepa)
            idx += BS

            with torch.autocast("cuda", dtype=AMP_DTYPE):
                # LM Pass
                out_lm = model(input_ids=batch["full_ids"], attention_mask=batch["full_mask"], labels=batch["full_lab"])
                lm_loss = out_lm.loss

                if need_jepa and batch["has_jepa"]:

                    out_ctx = model(input_ids=batch["ctx_ids"], attention_mask=batch["ctx_mask"], output_hidden_states=True)
                    z_ctx = out_ctx.hidden_states[args.layer][range(len(batch["ctx_last_idx"])), batch["ctx_last_idx"], :]


                    model.eval()
                    with torch.no_grad():
                        cm = model.disable_adapter() if args.target == "base" else contextlib.nullcontext()
                        with cm:
                            out_tgt = model(input_ids=batch["tgt_ids"], attention_mask=batch["tgt_mask"], output_hidden_states=True)
                            z_target = out_tgt.hidden_states[args.layer][range(len(batch["tgt_last_idx"])), batch["tgt_last_idx"], :].detach()
                    model.train()

                    c_std, c_pair = compute_anisotropy_stats(z_ctx)
                    t_std, t_pair = compute_anisotropy_stats(z_target)

                    ctx_std_accum += c_std / ACCUM
                    ctx_pair_accum += c_pair / ACCUM
                    tgt_std_accum += t_std / ACCUM
                    tgt_pair_accum += t_pair / ACCUM

                    # Cosine JEPA Loss
                    cos_sim = F.cosine_similarity(z_ctx, z_target, dim=-1)
                    jepa_loss = 1.0 - torch.mean(cos_sim)
                else:
                    jepa_loss = torch.zeros((), device="cuda")

                total_loss = (args.gamma * lm_loss) + (args.lbd * jepa_loss)

            scaler.scale(total_loss / ACCUM).backward()

            tot_loss_accum += total_loss.item() / ACCUM
            tot_lm_accum += lm_loss.item() / ACCUM
            tot_jepa_accum += jepa_loss.item() / ACCUM

        scaler.unscale_(opt)
        gn = torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
        opt.zero_grad(set_to_none=True)

        w_log.writerow([
            step,
            f"{tot_loss_accum:.6f}",
            f"{tot_lm_accum:.6f}",
            f"{tot_jepa_accum:.6f}",
            f"{ctx_std_accum:.6f}",
            f"{ctx_pair_accum:.6f}",
            f"{tgt_std_accum:.6f}",
            f"{tgt_pair_accum:.6f}",
            f"{lr:.8f}",
            f"{float(gn):.6f}"
        ])

        if step % 20 == 0:
            f_log.flush()
            print(f"Step {step}/{total_steps} | Loss: {tot_loss_accum:.4f} | LM: {tot_lm_accum:.4f} | JEPA: {tot_jepa_accum:.4f} | Tgt Cos: {tgt_pair_accum:.4f}")

        if step in ck:
            pct = ck[step]
            ckpt_dir = f"{OUT}/ckpt_{pct}pct"
            print(f"\n--> Saving Checkpoint at {pct}% to {ckpt_dir}...")
            model.save_pretrained(ckpt_dir)
            tok.save_pretrained(ckpt_dir)

    f_log.close()

    train_mins = (time.time() - t0) / 60.0
    peak_vram = torch.cuda.max_memory_allocated() / (1024**3)

    telemetry_data = {
        "arm_name": ARM_NAME,
        "training_type": "jepa" if need_jepa else "plain_lm",
        "k": args.k if need_jepa else None,
        "lambda_jepa": args.lbd,
        "gamma_lm": args.gamma,
        "layer": args.layer if need_jepa else None,
        "target_type": args.target if need_jepa else None,
        "pred_fix": args.pred_fix,
        "len_filter_counts_pred_token": True,
        "seed": args.seed,
        "precision": args.precision,
        "micro_batch_size": BS,
        "gradient_accumulation_steps": ACCUM,
        "effective_batch_size": BS * ACCUM,
        "max_len": MAX_LEN,
        "n_train_samples": len(data),
        "total_training_steps": total_steps,
        "train_minutes": round(train_mins, 2),
        "train_peak_vram_gb": round(peak_vram, 2),
        "trainable_params": sum(p.numel() for p in params)
    }
    telemetry_data.update(DATA_INFO)

    return telemetry_data



# MAIN
def main():
    configure()
    if os.path.exists(os.path.join(OUT, "ckpt_100pct")) and not args.overwrite:
        print(f"{OUT} already has a finished ckpt_100pct -> nothing to do (pass --overwrite to retrain).")
        sys.exit(0)
    os.makedirs(OUT, exist_ok=True)

    need_jepa = args.lbd > 0.0
    print(f"\n=======================================================")
    print(f" STARTING TRAINING: {ARM_NAME.upper()}")
    print(f" Base Model Path: {MODEL_DIR}")
    print(f" Output Dir     : {OUT}")
    if need_jepa:
        print(f" Method         : JEPA/alignment | k={args.k} | lbd={args.lbd} | Layer={args.layer} | Target={args.target} | pred_fix={args.pred_fix}")
    else:
        print(f" Method         : plain LM fine-tuning | replay_frac={args.replay_frac}")
    print(f" Seed={args.seed} | Batching: bs={BS} x accum={ACCUM} = {BS * ACCUM} | max_len={MAX_LEN} | precision={args.precision}")
    print(f"=======================================================\n")

    tok = AutoTokenizer.from_pretrained(MODEL_DIR)
    tok.pad_token = tok.pad_token or tok.eos_token

    PRED_ID = tok.convert_tokens_to_ids(args.pred_token)
    if need_jepa and args.k == 1:
        assert isinstance(PRED_ID, int) and tok.convert_ids_to_tokens(PRED_ID) == args.pred_token, f"{args.pred_token} is not a token of this tokenizer"

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_DIR,
        torch_dtype=AMP_DTYPE,
        device_map="cuda"
    )

    if args.pred_fix:

        with torch.no_grad():
            emb = model.get_input_embeddings().weight
            emb[PRED_ID] = emb.mean(dim=0, dtype=torch.float32).to(emb.dtype)
        print(f"[PRED] token {args.pred_token} (id {PRED_ID}): embedding row set to the mean embedding, and only that row is trainable")

    lora_kwargs = dict(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        task_type=TaskType.CAUSAL_LM,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    )
    if args.pred_fix:
        lora_kwargs["trainable_token_indices"] = [PRED_ID]
    peft_config = LoraConfig(**lora_kwargs)
    model = get_peft_model(model, peft_config)
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()
    model.print_trainable_parameters()  # expected 11,272,192 but 11,274,240 with --pred_fix ([PRED] becomes trainable)

    telemetry = train(model, tok, PRED_ID)

    telemetry_path = os.path.join(OUT, "telemetry.json")
    with open(telemetry_path, "w", encoding="utf-8") as f:
        json.dump(telemetry, f, indent=2)

    print("\n=======================================================")
    print(f"      TRAINING COMPLETED : {ARM_NAME}")
    print("=======================================================")
    print(f"Saved training log to    : {OUT}/train_log.csv")
    print(f"Saved 100% adapter to    : {OUT}/ckpt_100pct")
    print(f"Saved telemetry record to: {telemetry_path}")


if __name__ == "__main__":
    main()

## Setup 4/4: evaluation script (`eval.py`, one arm at a time)

In [ ]:
%%writefile eval.py
# ======================================================================================
# RUN FROM A COLAB CELL (one eval at a time):
#   !python eval.py --arm plain_fine_tuning --tests flores    # FLORES EN->VI and VI->EN (BLEU, chrF++)
#   !python eval.py --arm plain_fine_tuning --tests mlqa      # MLQA-vi (exact match, F1)
#   !python eval.py --arm plain_fine_tuning --tests ifeval    # IFEval (strict / loose)
#
# Extras:  --tests flores mlqa    several at once        --tests all   everything (default)
#          --arm base             the un-tuned model     --quick       8 examples, pipeline check
#          --force                redo finished tests    --n-flores N  random sample of N (default: all rows)
#
# Modular options (all optional; defaults reproduce the previous behaviour):
#          --ckpt NAME            which checkpoint folder inside the arm (default ckpt_100pct; falls back to the
#                                 arm folder itself if the adapter files are there)
#          --out-name NAME        results folder name inside the arm (default "eval")
#          --dtype fp16|bf16      generation precision (default fp16; use fp16 on a T4)
#          --base-model ID        base model (default meta-llama/Llama-3.2-1B-Instruct)
#          --seed N               FLORES sampling seed (default 42)
#          --bs-flores/--bs-mlqa/--bs-ifeval            generation batch sizes
#          --max-new-flores/--max-new-mlqa/--max-new-ifeval   max generated tokens
#          --flores-prompt short|train   FLORES instruction wording (default short = previous behaviour;
#                                        train = the wording the arms were trained with)
#
# ALL metrics go into ONE file:  <adapter-root>/<arm>/eval/results.json
# Each test is written into it the moment it finishes.
# ======================================================================================
"""
QUICK MODE is a somke test with only 8 examples per eval.

"""
import argparse
import glob
import hashlib
import importlib
import json
import os
import re
import string
import subprocess
import sys
import time
import types
import unicodedata
from collections import Counter

import pandas as pd
import peft
import sacrebleu
import torch
import transformers
from langdetect import DetectorFactory, LangDetectException, detect
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

DetectorFactory.seed = 0


ADAPTER_ROOT = "/content/drive/MyDrive/adapters"
DATA_DIR = "data"
BASE_MODEL = "meta-llama/Llama-3.2-1B-Instruct"
SEED = 42
BS = {"flores": 32, "mlqa": 16, "ifeval": 50}
MAX_NEW = {"flores": 200, "mlqa": 32, "ifeval": 1280}
DTYPES = {"fp16": torch.float16, "bf16": torch.bfloat16}
DTYPE_NAMES = {"fp16": "float16", "bf16": "bfloat16"}
DEFAULT_CKPT = "ckpt_100pct"
DEFAULT_OUT_NAME = "eval"

MLQA_PROMPT = (
    "Context: {context}\n\nQuestion: {question}\n\n"
    "Answer the question with a short phrase copied from the context. Do not explain."
)
FLORES_PROMPTS = {
    "short": {"en2vi": "Translate to Vietnamese: {}", "vi2en": "Translate to English: {}"},
    "train": {
        "en2vi": "Translate the following text to Vietnamese. Output only the translation.\n\n{}",
        "vi2en": "Translate the following text to English. Output only the translation.\n\n{}",
    },
}
QUICK_N = 8
SCHEMA = 2
TEST_NAMES = ("flores_en2vi", "flores_vi2en", "mlqa_vi", "ifeval")

ARM = None
N_FLORES = None
QUICK = False
FORCE = False
DTYPE_KEY = "fp16"
FLORES_PROMPT = "short"
ARM_DIR = None  # <adapter-root>/<arm>
ADAPTER_DIR = None
RESULT_DIR = None
RESULTS_PATH = None
model = tok = None



def save_jsonl(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        rows = [json.loads(line) for line in f]
    return rows[:QUICK_N] if QUICK else rows


def mean(xs):
    return sum(xs) / max(1, len(xs))


def pct(xs):
    return round(100 * mean([bool(x) for x in xs]), 2)


def ci95(xs):
    n = len(xs)
    if n < 2:
        return 0.0
    m = mean(xs)
    var = sum((x - m) ** 2 for x in xs) / (n - 1)
    return round(100 * 1.96 * (var / n) ** 0.5, 2)


def lang_of(text):
    try:
        return detect(text)
    except LangDetectException:
        return None


def gen_stats(outs, max_new):
    lens = [len(ids) for ids in tok(outs, add_special_tokens=False).input_ids]
    return {
        "avg_gen_tokens": round(mean(lens), 1),
        "truncated_pct": pct([n >= max_new - 2 for n in lens]),
        "empty_pct": pct([not o.strip() for o in outs]),
    }


def pct_table(counts):
    """{key: [n, strict_hits, loose_hits]} -> {key: {"n", "strict", "loose"}} percent."""
    return {k: {"n": n, "strict": round(100 * s / n, 2), "loose": round(100 * l / n, 2)}
            for k, (n, s, l) in sorted(counts.items())}


def load_results():
    if RESULTS_PATH and os.path.exists(RESULTS_PATH):
        with open(RESULTS_PATH, encoding="utf-8") as f:
            return json.load(f)
    return {}


def save_results(results):
    #for atomic write
    results["last_updated"] = time.strftime("%Y-%m-%d %H:%M")
    tmp = RESULTS_PATH + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    os.replace(tmp, RESULTS_PATH)


def run_info():
    info = {
        "arm": ARM, "base_model": BASE_MODEL, "quick_mode": QUICK, "seed": SEED,
        "generation": {"decoding": "greedy", "dtype": DTYPE_NAMES[DTYPE_KEY], "max_new_tokens": dict(MAX_NEW),
                       "batch_size": dict(BS), "flores_prompt": FLORES_PROMPT},
        "flores_sample": "all rows" if N_FLORES is None else f"random {N_FLORES} (seed {SEED})",
        "environment": {
            "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
            "transformers": transformers.__version__, "peft": peft.__version__, "sacrebleu": sacrebleu.__version__,
        },
    }
    if ARM != "base":
        info["checkpoint"] = os.path.relpath(ADAPTER_DIR, ARM_DIR)
        cfg = os.path.join(ADAPTER_DIR, "adapter_config.json")
        if os.path.exists(cfg):
            with open(cfg, encoding="utf-8") as f:
                c = json.load(f)
            info["lora"] = {k: c.get(k) for k in ("r", "lora_alpha", "lora_dropout", "target_modules", "use_rslora", "use_dora")}
    return info



def load_model():
    global model, tok
    assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU."
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=DTYPES[DTYPE_KEY]).to("cuda")
    if ARM != "base":
        assert os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json")), (
            f"No adapter found in {ADAPTER_DIR} (expected adapter_config.json + adapter_model.safetensors). "
            f"Use --ckpt to pick a checkpoint folder inside {ARM_DIR}."
        )
        model = PeftModel.from_pretrained(model, ADAPTER_DIR)
        model = model.merge_and_unload()
        print(f"loaded base + adapter: {ADAPTER_DIR}")
    else:
        print("evaluating the BASE model (no adapter)")
    model.eval()


@torch.no_grad()
def generate(prompts, max_new_tokens, bs, partial_path=None):
    texts = [
        tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True)
        for p in prompts
    ]
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]), reverse=True)
    outs = [None] * len(texts)
    key = hashlib.md5(json.dumps([prompts, max_new_tokens], ensure_ascii=False).encode("utf-8")).hexdigest()
    if partial_path and os.path.exists(partial_path) and not FORCE:
        with open(partial_path, encoding="utf-8") as f:
            part = json.load(f)
        if part.get("key") == key:
            for i, o in part["outs"].items():
                outs[int(i)] = o
            print(f"   resuming: {sum(o is not None for o in outs)}/{len(outs)} already generated ({partial_path})")
    t0 = time.time()
    for s in range(0, len(order), bs):
        idx = order[s: s + bs]
        if all(outs[i] is not None for i in idx):
            continue
        enc = tok([texts[i] for i in idx], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        out = model.generate(
            **enc, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None,
            pad_token_id=tok.pad_token_id,
        )
        for j, i in enumerate(idx):
            outs[i] = tok.decode(out[j, enc.input_ids.shape[1]:], skip_special_tokens=True).strip()
        if partial_path:
            tmp = partial_path + ".tmp"
            with open(tmp, "w", encoding="utf-8") as f:
                json.dump({"key": key, "outs": {str(i): o for i, o in enumerate(outs) if o is not None}}, f, ensure_ascii=False)
            os.replace(tmp, partial_path)
        print(f"   generated {min(s + bs, len(order))}/{len(order)}  ({time.time() - t0:.0f}s)")
    return outs


def generate_cached(name, prompts, max_new_tokens, bs):
    path = os.path.join(RESULT_DIR, f"{name}_generations.json")
    if os.path.exists(path) and not FORCE:
        with open(path, encoding="utf-8") as f:
            outs = json.load(f)
        if len(outs) == len(prompts):
            print(f"   reusing saved generations: {path}")
            return outs
    partial = path.replace("_generations.json", "_generations.partial.json")
    outs = generate(prompts, max_new_tokens, bs, partial_path=partial)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(outs, f, ensure_ascii=False)
    if os.path.exists(partial):
        os.remove(partial)
    return outs



PUNCT = {chr(i) for i in range(sys.maxunicode) if unicodedata.category(chr(i)).startswith("P")}.union(string.punctuation)


def norm_vi(s):
    s = "".join(ch for ch in s.lower() if ch not in PUNCT)
    s = re.sub(r"\b(của|là|cái|chiếc|những)\b", " ", s)
    return " ".join(s.split())


def f1_score(pred, gold):
    p, g = norm_vi(pred).split(), norm_vi(gold).split()
    same = sum((Counter(p) & Counter(g)).values())
    if same == 0:
        return 0.0
    prec, rec = same / len(p), same / len(g)
    return 2 * prec * rec / (prec + rec)


def em_score(pred, gold):
    return float(norm_vi(pred) == norm_vi(gold))



def setup_ifeval_checker():
    root = "/content/lm_eval_ifeval"
    if not os.path.exists(f"{root}/lm_eval/tasks/ifeval/utils.py"):
        subprocess.run([sys.executable, "-m", "pip", "download", "lm_eval==0.4.13", "--no-deps",
                        "-d", "/content/lme_whl", "-q"], check=True)
        import zipfile
        whl = glob.glob("/content/lme_whl/lm_eval-*.whl")[0]
        with zipfile.ZipFile(whl) as z:
            for n in z.namelist():
                if n.startswith("lm_eval/tasks/ifeval/") and n.endswith(".py"):
                    z.extract(n, root)
    for name, sub in [("lm_eval", "lm_eval"), ("lm_eval.tasks", "lm_eval/tasks"),
                      ("lm_eval.tasks.ifeval", "lm_eval/tasks/ifeval")]:
        m = types.ModuleType(name)
        m.__path__ = [f"{root}/{sub}"]
        sys.modules[name] = m
    return importlib.import_module("lm_eval.tasks.ifeval.utils")



def load_flores():
    jpath = os.path.join(DATA_DIR, "flores.json")  # list of {"id", "en", "vi"}
    if os.path.exists(jpath):
        with open(jpath, encoding="utf-8") as f:
            df = pd.DataFrame(json.load(f))
    else:
        df = pd.read_csv(os.path.join(DATA_DIR, "flores.csv"))
    df = df.dropna(subset=["en", "vi"])
    if N_FLORES is not None and len(df) > N_FLORES:
        df = df.sample(n=N_FLORES, random_state=SEED)
    df = df.reset_index(drop=True)
    return df.head(QUICK_N) if QUICK else df


def run_flores(direction):
    df = load_flores()
    tmpl = FLORES_PROMPTS[FLORES_PROMPT][direction]
    if direction == "en2vi":
        src, ref, target = list(df.en), list(df.vi), "vi"
    else:
        src, ref, target = list(df.vi), list(df.en), "en"
    hyp = generate_cached(f"flores_{direction}", [tmpl.format(s) for s in src], MAX_NEW["flores"], BS["flores"])
    bleu = sacrebleu.corpus_bleu(hyp, [ref])
    chrf = sacrebleu.corpus_chrf(hyp, [ref], word_order=2)
    langs = [lang_of(h) for h in hyp]
    metrics = {
        "n": len(hyp),
        "bleu": round(bleu.score, 2),
        "chrfpp": round(chrf.score, 2),
        "bleu_length_ratio": round(bleu.ratio, 3),  # reference length
        "bleu_brevity_penalty": round(bleu.bp, 3),
        "target_language_pct": pct([lang == target for lang in langs]),  # output is in the requested language
        "copied_source_pct": pct([h.strip() == s.strip() for h, s in zip(hyp, src)]),  # model just repeated the input
        "multiline_pct": pct(["\n" in h for h in hyp]),  # extra chatter
    }
    metrics.update(gen_stats(hyp, MAX_NEW["flores"]))
    rows = [{"id": int(i), "source": s, "reference": r, "hypothesis": h, "lang": lang,
             "chrfpp": round(sacrebleu.sentence_chrf(h, [r], word_order=2).score, 2)}
            for i, s, r, h, lang in zip(df.id, src, ref, hyp, langs)]
    return metrics, rows


def run_mlqa():
    rows = load_jsonl(os.path.join(DATA_DIR, "mlqa_vi.jsonl"))
    prompts = [MLQA_PROMPT.format(context=r["context"], question=r["question"]) for r in rows]
    outs = generate_cached("mlqa_vi", prompts, MAX_NEW["mlqa"], BS["mlqa"])
    ems, f1s, in_ctx, pred_words, gold_words, out_rows = [], [], [], [], [], []
    for r, o in zip(rows, outs):
        pred = o.split("\n")[0].strip()
        golds = list(r["answers"]["text"])
        em, f1 = max(em_score(pred, g) for g in golds), max(f1_score(pred, g) for g in golds)
        p_norm = norm_vi(pred)
        found = bool(p_norm) and p_norm in norm_vi(r["context"])
        ems.append(em)
        f1s.append(f1)
        in_ctx.append(found)
        pred_words.append(len(pred.split()))
        gold_words.append(len(golds[0].split()))
        out_rows.append({"id": r.get("id"), "question": r["question"], "gold": golds, "prediction": pred,
                         "em": em, "f1": f1, "in_context": found})
    metrics = {
        "n": len(rows),
        "exact_match": round(100 * mean(ems), 2), "exact_match_ci95": ci95(ems),
        "f1": round(100 * mean(f1s), 2), "f1_ci95": ci95(f1s),
        "answer_in_context_pct": pct(in_ctx),
        "avg_pred_words": round(mean(pred_words), 1), "avg_gold_words": round(mean(gold_words), 1),
        "multiline_pct": pct(["\n" in o.strip() for o in outs]),  # model explained instead of answering briefly
    }
    metrics.update(gen_stats(outs, MAX_NEW["mlqa"]))
    return metrics, out_rows


def fix_ints(kwargs):
    return [{k: (int(v) if isinstance(v, float) and v.is_integer() else v) for k, v in d.items()} for d in kwargs]


def run_ifeval():
    utils = setup_ifeval_checker()
    rows = load_jsonl(os.path.join(DATA_DIR, "ifeval.jsonl"))
    outs = generate_cached("ifeval", [r["prompt"] for r in rows], MAX_NEW["ifeval"], BS["ifeval"])
    ps, pl, ins, inl, out_rows, errors = [], [], [], [], [], 0
    by_id, by_cat, langs = {}, {}, []
    for r, o in zip(rows, outs):
        doc = dict(r, kwargs=fix_ints(r["kwargs"]))
        try:
            res = utils.process_results(doc, [o])
        except Exception as e:
            errors += 1
            print(f"   scoring error on key {r['key']}: {type(e).__name__}: {e}")
            n_inst = len(r["instruction_id_list"])
            res = {"prompt_level_strict_acc": False, "prompt_level_loose_acc": False,
                   "inst_level_strict_acc": [False] * n_inst, "inst_level_loose_acc": [False] * n_inst}
        ps.append(res["prompt_level_strict_acc"])
        pl.append(res["prompt_level_loose_acc"])
        ins += res["inst_level_strict_acc"]
        inl += res["inst_level_loose_acc"]

        for iid, s, l in zip(r["instruction_id_list"], res["inst_level_strict_acc"], res["inst_level_loose_acc"]):
            for table, key in ((by_id, iid), (by_cat, iid.split(":")[0])):
                c = table.setdefault(key, [0, 0, 0])
                c[0] += 1
                c[1] += int(s)
                c[2] += int(l)

        lang = None
        if "language:response_language" not in r["instruction_id_list"]:
            lang = lang_of(o)
            if lang:
                langs.append(lang)
        out_rows.append({"key": r["key"], "prompt": r["prompt"], "response": o, "lang": lang,
                         "strict": bool(res["prompt_level_strict_acc"]), "loose": bool(res["prompt_level_loose_acc"])})
    metrics = {
        "n": len(rows),
        "prompt_strict": round(100 * mean(ps), 2), "prompt_strict_ci95": ci95([float(x) for x in ps]),
        "prompt_loose": round(100 * mean(pl), 2), "prompt_loose_ci95": ci95([float(x) for x in pl]),
        "inst_strict": round(100 * mean(ins), 2), "inst_loose": round(100 * mean(inl), 2),
        "non_english_pct": pct([lg != "en" for lg in langs]),  # English prompts answered in another language
        "vietnamese_pct": pct([lg == "vi" for lg in langs]),  # forgetting / drift in Vietnamese
        "scoring_errors": errors,
    }
    metrics.update(gen_stats(outs, MAX_NEW["ifeval"]))
    metrics["by_category"] = pct_table(by_cat)
    metrics["by_instruction"] = pct_table(by_id)
    if errors:
        print(f"   WARNING: {errors} responses could not be scored and were counted as failures")
    return metrics, out_rows


def cached(name, fn, expect_n=None):
    """Run one test and append its metrics to results.json. Skipped if a current-format result already exists."""
    saved = load_results().get(name)
    if saved is not None and not FORCE:
        if saved.get("schema") != SCHEMA:
            print(f"[{name}] saved result is from an older version of this script -> rescoring (saved generations are reused)")
        elif expect_n is not None and saved.get("n") != expect_n:
            print(f"[{name}] saved result has n={saved.get('n')} but n={expect_n} is needed -> rerunning")
        else:
            print(f"[{name}] already done -> loaded from {RESULTS_PATH}")
            return saved
    print(f"\n[{name}] running ...")
    t0 = time.time()
    metrics, rows = fn()
    metrics["seconds"] = round(time.time() - t0)
    metrics["finished_at"] = time.strftime("%Y-%m-%d %H:%M")
    metrics["schema"] = SCHEMA
    save_jsonl(rows, os.path.join(RESULT_DIR, f"{name}_predictions.jsonl"))
    results = load_results()
    results[name] = metrics
    save_results(results)
    short = {k: v for k, v in metrics.items() if not isinstance(v, dict)}
    print(f"[{name}] {short}")
    print(f"   saved to {RESULTS_PATH}")
    print("   example output:", repr(str(rows[0].get("hypothesis") or rows[0].get("prediction") or rows[0].get("response"))[:150]))
    return metrics


#summary on CLI
def print_summary():
    s = load_results()
    print("\n================ SUMMARY:", ARM, "(QUICK MODE)" if QUICK else "", "================")
    for k in TEST_NAMES:
        m = s.get(k)
        if m is None:
            continue
        if m.get("schema") != SCHEMA:
            print(f"{k:14s} (old format: rerun this test to refresh it)")
        elif k.startswith("flores"):
            print(f"{k:14s} BLEU {m['bleu']:6.2f}   chrF++ {m['chrfpp']:6.2f}   right language {m['target_language_pct']:5.1f}%   "
                  f"length ratio {m['bleu_length_ratio']:.2f}   (n={m['n']})")
        elif k == "mlqa_vi":
            print(f"{k:14s} EM   {m['exact_match']:6.2f} +/-{m['exact_match_ci95']:.1f}   F1 {m['f1']:6.2f} +/-{m['f1_ci95']:.1f}   "
                  f"answer in context {m['answer_in_context_pct']:5.1f}%   (n={m['n']})")
        else:
            print(f"{k:14s} prompt strict {m['prompt_strict']:.2f} +/-{m['prompt_strict_ci95']:.1f} / loose {m['prompt_loose']:.2f}   "
                  f"inst strict {m['inst_strict']:.2f} / loose {m['inst_loose']:.2f}   "
                  f"non-English {m['non_english_pct']:.1f}%   truncated {m['truncated_pct']:.1f}%   (n={m['n']})")
    print("saved to", RESULTS_PATH)


def parse_args(argv=None):
    p = argparse.ArgumentParser(description="Evaluate ONE arm (LoRA adapter folder or the base model).")
    p.add_argument("--arm", required=True,
                   help='adapter folder name inside --adapter-root, or "base" for the un-tuned model')
    p.add_argument("--tests", nargs="+", default=["all"], choices=["flores", "mlqa", "ifeval", "all"],
                   help="which evals to run (default: all)")
    p.add_argument("--quick", action="store_true", help="8 examples per test, saved to <out-name>_quick/")
    p.add_argument("--force", action="store_true", help="redo tests even if results already exist")
    p.add_argument("--n-flores", type=int, default=None,
                   help="FLORES sentences per direction (default: all rows in flores.json)")
    p.add_argument("--adapter-root", default=ADAPTER_ROOT)
    p.add_argument("--data-dir", default=DATA_DIR)
    p.add_argument("--ckpt", default=DEFAULT_CKPT,
                   help=f"checkpoint folder inside the arm (default {DEFAULT_CKPT}); if it does not exist, the arm "
                        "folder itself is used when the adapter files are there. Ignored for --arm base.")
    p.add_argument("--out-name", default=None,
                   help=f'results folder name inside the arm (default "{DEFAULT_OUT_NAME}"; "{DEFAULT_OUT_NAME}_<ckpt>" '
                        f"when you evaluate a checkpoint other than {DEFAULT_CKPT})")
    p.add_argument("--dtype", choices=["fp16", "bf16"], default="fp16",
                   help="generation precision (default fp16, which is the fast one on a T4)")
    p.add_argument("--base-model", default=BASE_MODEL)
    p.add_argument("--seed", type=int, default=SEED, help="seed for the FLORES random sample (with --n-flores)")
    p.add_argument("--bs-flores", type=int, default=BS["flores"])
    p.add_argument("--bs-mlqa", type=int, default=BS["mlqa"])
    p.add_argument("--bs-ifeval", type=int, default=BS["ifeval"])
    p.add_argument("--max-new-flores", type=int, default=MAX_NEW["flores"])
    p.add_argument("--max-new-mlqa", type=int, default=MAX_NEW["mlqa"])
    p.add_argument("--max-new-ifeval", type=int, default=MAX_NEW["ifeval"])
    p.add_argument("--flores-prompt", choices=list(FLORES_PROMPTS), default="short",
                   help='FLORES instruction wording: "short" (previous behaviour) or "train" (the wording the arms '
                        "were trained with). Use the SAME value for every arm you compare, including base.")
    return p.parse_args(argv)


def resolve_adapter_dir(arm_dir, ckpt):
    if ckpt and os.path.exists(os.path.join(arm_dir, ckpt, "adapter_config.json")):
        return os.path.join(arm_dir, ckpt)
    if os.path.exists(os.path.join(arm_dir, "adapter_config.json")):
        return arm_dir
    return os.path.join(arm_dir, ckpt) if ckpt else arm_dir


def setup_run(args):
    global ARM, ADAPTER_ROOT, DATA_DIR, BASE_MODEL, SEED, N_FLORES, QUICK, FORCE
    global DTYPE_KEY, FLORES_PROMPT, ARM_DIR, ADAPTER_DIR, RESULT_DIR, RESULTS_PATH
    ARM, ADAPTER_ROOT, DATA_DIR, BASE_MODEL, SEED = args.arm, args.adapter_root, args.data_dir, args.base_model, args.seed
    N_FLORES, QUICK, FORCE = args.n_flores, args.quick, args.force
    DTYPE_KEY, FLORES_PROMPT = args.dtype, args.flores_prompt
    BS.update(flores=args.bs_flores, mlqa=args.bs_mlqa, ifeval=args.bs_ifeval)
    MAX_NEW.update(flores=args.max_new_flores, mlqa=args.max_new_mlqa, ifeval=args.max_new_ifeval)

    ARM_DIR = os.path.join(ADAPTER_ROOT, ARM)
    ADAPTER_DIR = ARM_DIR if ARM == "base" else resolve_adapter_dir(ARM_DIR, args.ckpt)
    folder = os.path.basename(ADAPTER_DIR)
    default_ckpt = ARM == "base" or ADAPTER_DIR == ARM_DIR or folder == DEFAULT_CKPT
    out_name = args.out_name or (DEFAULT_OUT_NAME if default_ckpt else f"{DEFAULT_OUT_NAME}_{folder}")
    RESULT_DIR = os.path.join(ARM_DIR, out_name + ("_quick" if QUICK else ""))
    RESULTS_PATH = os.path.join(RESULT_DIR, "results.json")
    return ["flores", "mlqa", "ifeval"] if "all" in args.tests else args.tests


def main():
    tests = setup_run(parse_args())

    os.makedirs(RESULT_DIR, exist_ok=True)
    load_model()


    info = run_info()
    prev = load_results().get("generation")
    if prev and not FORCE:
        changed = [k for k in ("decoding", "dtype", "max_new_tokens", "flores_prompt")
                   if k in prev and prev[k] != info["generation"][k]]
        if changed:
            raise SystemExit(
                f"{RESULTS_PATH} was made with different settings ({', '.join(changed)}). "
                "Use --force to redo the tests, or --out-name to write to a new folder."
            )

    results = load_results()
    results.pop("date", None)
    results.update(info)
    save_results(results)

    if "flores" in tests:
        n_flores = len(load_flores())
        cached("flores_en2vi", lambda: run_flores("en2vi"), expect_n=n_flores)
        cached("flores_vi2en", lambda: run_flores("vi2en"), expect_n=n_flores)
    if "mlqa" in tests:
        cached("mlqa_vi", run_mlqa)
    if "ifeval" in tests:
        cached("ifeval", run_ifeval)

    print_summary()


if __name__ == "__main__":
    main()

<br><br>


## Base model
Only an eval cell, there is nothing to train. Results go to `adapters/base/eval/`


In [ ]:
!python eval.py --arm base --adapter-root {ADAPTER_ROOT} --data-dir {DATA_DIR}

## Plain fine-tuning
Plain next-token (LM) loss only. The baseline every other arm is compared with.

**Cell 1 = train, cell 2 = eval**.

Output: `adapters/plain_fine_tuning/` (checkpoints, `train_log.csv`, `telemetry.json`) and `adapters/plain_fine_tuning/eval/results.json`.

In [ ]:
!python train.py --arm_name plain_fine_tuning --lbd 0 --k 0 \
    --seed 42 --bs 4 --accum 4 --max_len 256 --n_total 50000 --lr 2e-4 --precision fp16 --out_root {ADAPTER_ROOT}

In [ ]:
!python eval.py --arm plain_fine_tuning --adapter-root {ADAPTER_ROOT} --data-dir {DATA_DIR}

## Plain alignment
LM loss + a cosine alignment loss between the Vietnamese and English last-token vectors, λ=0.5.

 Output: `adapters/plain_alignment/` and `adapters/plain_alignment/eval/results.json`.

In [ ]:
!python train.py --arm_name plain_alignment --lbd 0.5 --k 0 --target live --layer -1 --gamma 1.0 \
    --seed 42 --bs 4 --accum 4 --max_len 256 --n_total 50000 --lr 2e-4 --precision fp16 --out_root {ADAPTER_ROOT}

In [ ]:
!python eval.py --arm plain_alignment --adapter-root {ADAPTER_ROOT} --data-dir {DATA_DIR}

## JEPA live (k=1)
k=1 (1 [PRED] token), live target. λ=0.5

Output: `adapters/jepa_live/` and `adapters/jepa_live/eval/results.json`.

In [ ]:
!python train.py --arm_name jepa_live --lbd 0.5 --k 1 --target live --layer -1 --gamma 1.0 \
    --seed 42 --bs 4 --accum 4 --max_len 256 --n_total 50000 --lr 2e-4 --precision fp16 --out_root {ADAPTER_ROOT}

In [ ]:
!python eval.py --arm jepa_live --adapter-root {ADAPTER_ROOT} --data-dir {DATA_DIR}

## JEPA anchored (k=0)
k=0, λ=0.5

Output: `adapters/jepa_anchored_k0/` and `adapters/jepa_anchored_k0/eval/results.json`.

In [ ]:
!python train.py --arm_name jepa_anchored_k0 --lbd 0.5 --k 0 --target base --layer -1 --gamma 1.0 \
    --seed 42 --bs 4 --accum 4 --max_len 256 --n_total 50000 --lr 2e-4 --precision fp16 --out_root {ADAPTER_ROOT}

In [ ]:
!python eval.py --arm jepa_anchored_k0 --adapter-root {ADAPTER_ROOT} --data-dir {DATA_DIR}

## JEPA anchored (k=1)
k=1 (1 [PRED] token), anchored target, λ=0.5.

Output: `adapters/jepa_anchored/` and `adapters/jepa_anchored/eval/results.json`.

In [ ]:
!python train.py --arm_name jepa_anchored --lbd 0.5 --k 1 --target base --layer -1 --gamma 1.0 \
    --seed 42 --bs 4 --accum 4 --max_len 256 --n_total 50000 --lr 2e-4 --precision fp16 --out_root {ADAPTER_ROOT}

In [ ]:
!python eval.py --arm jepa_anchored --adapter-root {ADAPTER_ROOT} --data-dir {DATA_DIR}

## JEPA anchored (k=1, trainable [PRED] token)
Same as anchored k=1, but the [PRED] embedding row starts at the mean embedding and only that row is trainable.

Output: `adapters/jepa_anchored_new_pred/` and `adapters/jepa_anchored_new_pred/eval/results.json`.

In [ ]:
!python train.py --arm_name jepa_anchored_new_pred --lbd 0.5 --k 1 --target base --pred_fix --layer -1 --gamma 1.0 \
    --seed 42 --bs 4 --accum 4 --max_len 256 --n_total 50000 --lr 2e-4 --precision fp16 --out_root {ADAPTER_ROOT}

In [ ]:
!python eval.py --arm jepa_anchored_new_pred --adapter-root {ADAPTER_ROOT} --data-dir {DATA_DIR}

## Replay 10% (45k OPUS + 5k Alpaca)
Plain LM loss on 45,000 OPUS pairs + 5,000 `yahma/alpaca-cleaned` rows.

Output: `adapters/replay_10pct/` and `adapters/replay_10pct/eval/results.json`.

In [ ]:
!python train.py --arm_name replay_10pct --lbd 0 --k 0 --replay_frac 0.10 \
    --seed 42 --bs 4 --accum 4 --max_len 256 --n_total 50000 --lr 2e-4 --precision fp16 --out_root {ADAPTER_ROOT}

In [ ]:
!python eval.py --arm replay_10pct --adapter-root {ADAPTER_ROOT} --data-dir {DATA_DIR}

## Replay 10% + JEPA anchored (k=1)
Same 45,000 OPUS + 5,000 Alpaca mix as the replay arm, plus the anchored JEPA loss. The JEPA loss is computed on the OPUS rows of each batch only; Alpaca rows get the LM loss only.

Output: `adapters/replay_10pct_anchor_jepa/` and `adapters/replay_10pct_anchor_jepa/eval/results.json`.

In [ ]:
!python train.py --arm_name replay_10pct_anchor_jepa --lbd 0.5 --k 1 --target base --replay_frac 0.10 --layer -1 --gamma 1.0 \
    --seed 42 --bs 4 --accum 4 --max_len 256 --n_total 50000 --lr 2e-4 --precision fp16 --out_root {ADAPTER_ROOT}

In [ ]:
!python eval.py --arm replay_10pct_anchor_jepa --adapter-root {ADAPTER_ROOT} --data-dir {DATA_DIR}